Pruning Mask Generation for AdaptViT
==========================================
Extends SnapViT (https://github.com/WalterSimoncini/SnapViT, NeurIPS 2025)
with hardware-aligned group-of-8 MLP pruning and runs structured
pruning on pretrained ViT models to generate
binary pruning masks for downstream TVM compilation and RISC-V deployment.

Pipeline:
  1. Clone SnapViT and install dependencies
  2. Download Oxford-IIIT Pets as calibration data
  3. Patch three SnapViT source files with our modifications (described per-cell)
  4. Run GA-based pruning across multiple sparsity levels
  5. Save pruning masks to Drive

Author: Vishnu PS

## Clone SnapViT Repo and install dependencies

In [ ]:
import os
import shutil
import subprocess
from google.colab import drive

## Configuration and Envrironment setup

In [17]:
!export NUM_PROC=8
!export CACHE_DIR=/content/cache

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

BASE_PATH = f'/content/drive/MyDrive/PhD_Works/SnapViT_Q2/model_outputs'
base_dir = "/content/pruning/artifacts/ga"

!git clone https://github.com/WalterSimoncini/SnapViT.git
%cd SnapViT
!pip install -r requirements.txt

# Prepare Oxford-IIIT Pets calibration dataset

In [ ]:
%%writefile ./jobs/datasets/pets.sh
NUM_PROC=8
CACHE_DIR=/content/cache

# Remove the data directory if it exists
rm -rf $CACHE_DIR/pets
mkdir $CACHE_DIR/pets

# Download the images and annotations
wget -P $CACHE_DIR/pets https://thor.robots.ox.ac.uk/~vgg/data/pets/images.tar.gz
wget -P $CACHE_DIR/pets https://thor.robots.ox.ac.uk/~vgg/data/pets/annotations.tar.gz

# Unpack the images and annotations
tar -xf $CACHE_DIR/pets/images.tar.gz -C $CACHE_DIR/pets
tar -xf $CACHE_DIR/pets/annotations.tar.gz -C $CACHE_DIR/pets

# Generate the dataset
python pets.py \
    --num-proc 18 \
    --cache-dir $CACHE_DIR \
    --root-dir $CACHE_DIR/pets/ \
    --output-folder $CACHE_DIR/pets-hf

In [ ]:
!chmod +x /jobs/datasets/pets.sh
!./jobs/datasets/pets.sh

## Patching SnapViT files

## prune.py : Main entry point for SnapViT pruning pipeline.

Changes:
- Added `save_masks_path` parameter to the prune() call in the final evaluation loop.
- Ensured directory creation happens before pruning so masks can be saved reliably.
- Enhanced model saving to store full pruned model object (not just state_dict).
- Added detailed logging for hidden dims and heads after pruning.

In [ ]:
%cd ../

In [ ]:
# @title
%%writefile SnapViT/prune.py
import os
import json
import time
import copy
import torch
import logging
import argparse
import numpy as np
import torch.nn as nn

from typing import List
from torch.utils.data import DataLoader

from src.utils.ga.loss import GALossType
from src.datasets.fast import FastDataset
from src.models.enums import MLPLayerType
from src.utils.models import deepcopy_model
from src.models import load_model, ModelType
from src.models.prunable import PrunableModel
from src.utils.eval import eval_model_datasets
from src.inference import export_importance_scores
from src.models.prunable import load_prunable_model
from src.utils.performance import estimate_model_flops
from src.utils.ga.fitness import build_fitness_function
from src.utils.ga import GAOptimizerType, optimize_function_ga
from src.models.enums import PrunableModelType, SparseGPTDampingStrategy
from src.utils.logging import configure_logging, save_run_info, save_pruned_model
from src.datasets import load_dataset, load_eval_datasets, DatasetSplit, DatasetType
from src.utils.misc import get_device, seed_everything, default_cache_dir, pad_vector_to_match
from src.utils.models import count_parameters, count_mlp_parameters, count_attn_parameters, freeze_model


def main(args: argparse.Namespace):
    device = get_device()
    run_timestamp = int(time.time())
    output_dir = os.path.join(args.output_dir, f"run-{args.model_type.value}-{run_timestamp}")

    os.makedirs(output_dir)
    logging.info(f"the output directory is: {output_dir}")

    seed_everything(args.seed)

    model, transform = load_model(
        type_=args.model_type,
        cache_dir=args.cache_dir,
        device=device
    )

    num_blocks = len(model.blocks)
    num_heads = model.blocks[0].attn.num_heads

    if hasattr(model.blocks[0].mlp, MLPLayerType.FC1.value):
        hidden_dim = model.blocks[0].mlp.fc1.weight.shape[0]
    elif hasattr(model.blocks[0].mlp, MLPLayerType.W1.value):
        hidden_dim = model.blocks[0].mlp.w1.weight.shape[0]
    else:
        raise ValueError(f"Unknown MLP architecture: {model.blocks[0].mlp}")

    num_total_params = count_parameters(model)
    num_mlp_params = count_mlp_parameters(model=model)
    num_attn_params = count_attn_parameters(model=model)

    num_base_flops = estimate_model_flops(model=model)

    teacher = copy.deepcopy(model).to(device)

    if hasattr(teacher, "head"):
        teacher.head = nn.Identity()

    if hasattr(teacher, "head_dist"):
        teacher.head_dist = nn.Identity()

    freeze_model(model=teacher)

    gradient_layers = []

    for i in range(num_blocks):
        if hasattr(model.blocks[i].mlp, MLPLayerType.FC1.value):
            gradient_layers.append(f"blocks.{i}.mlp.fc1")
        elif hasattr(model.blocks[i].mlp, MLPLayerType.W1.value):
            gradient_layers.append(f"blocks.{i}.mlp.w1")

        gradient_layers.append(f"blocks.{i}.attn.qkv")

    freeze_model(model=model, exclusions=gradient_layers)

    logging.info(f"loaded model of type {args.model_type}")

    logging.info(f"the model has {num_blocks} blocks")
    logging.info(f"the model has {num_total_params} parameters")
    logging.info(f"the model has {num_base_flops} FLOPs")

    logging.info(f"the mlp has {num_mlp_params} parameters")
    logging.info(f"the attention blocks have {num_attn_params} parameters")

    logging.info(f"the mlp hidden dimensionality is {hidden_dim}")
    logging.info(f"the model has {num_heads} heads")

    logging.info(f"preserving at least {args.min_hidden_dim_keep_ratio * 100:.2f}% of the mlp hidden dimension in each block")
    logging.info(f"preserving at least {args.min_head_keep_ratio * 100:.2f}% of the heads in each block's attention layer")

    model_config = {
      "mlp_pruning_ratios": args.ga_mlp_pruning_ratios,
      "head_pruning_ratios": args.ga_heads_pruning_ratios,
      "min_hidden_dim_keep_ratio": args.min_hidden_dim_keep_ratio,
      "min_head_keep_ratio": args.min_head_keep_ratio,
      "ga_optimizer": args.ga_optimizer,
      "ga_batch_size": args.ga_batch_size,
      "ga_max_function_evaluations": args.ga_max_function_evaluations,
      "ga_num_pca_components": args.ga_num_pca_components,
      "ga_loss_types": args.ga_loss_types,
      "ga_max_eval_samples": args.ga_max_eval_samples,
    }

    for i, (mlp_ratio, head_ratio) in enumerate(zip(args.ga_mlp_pruning_ratios, args.ga_heads_pruning_ratios)):
        num_pruned_params = int(num_mlp_params * mlp_ratio + num_attn_params * head_ratio)
        total_pruning_ratio = round(num_pruned_params / num_total_params * 100, 2)

        logging.info(f"model {i}: pruning the mlp with ratio {mlp_ratio} and the heads with ratio {head_ratio}")
        logging.info(f"model {i}: pruning {num_pruned_params} parameters, i.e. {(total_pruning_ratio):.2f}% of the model")

    model, pruning_transform = load_prunable_model(
        type_=args.pruning_strategy,
        backbone=model,
        backbone_transform=transform,
        device=device,
        estimation_epochs=args.num_estimation_epochs,
        min_hidden_dim_keep_ratio=args.min_hidden_dim_keep_ratio,
        min_head_keep_ratio=args.min_head_keep_ratio
    )

    data_loader_args = {
        "batch_size": args.batch_size,
        "num_workers": args.num_workers
    }

    eval_datasets = load_eval_datasets(
        dataset_types=args.eval_datasets,
        cache_dir=args.cache_dir,
        transform=transform,
        max_knn_train_samples=args.max_knn_train_samples,
        **data_loader_args
    )

    pruning_dataset = load_dataset(
        type_=args.pruning_dataset,
        split=DatasetSplit.TRAIN,
        cache_dir=args.cache_dir,
        transform=pruning_transform,
        max_samples=args.max_samples
    )

    logging.info(f"loaded pruning dataset of type {args.pruning_dataset} with {len(pruning_dataset)} samples")
    logging.info(f"the pruning dataset has {len(pruning_dataset)} samples")

    pruning_data_loader = DataLoader(dataset=pruning_dataset, **data_loader_args)

    run_metrics = {
        "pruned": [],
        "baseline": {
            "hidden_dim": hidden_dim,
            "num_heads": num_heads,
            "flops": num_base_flops
        }
    }

    if args.eval_baseline:
        baseline_features_output_dir = os.path.join(output_dir, "baseline", "features")

        if args.save_features:
            logging.info(f"saving the features of the baseline model to {baseline_features_output_dir}")
            os.makedirs(baseline_features_output_dir, exist_ok=True)

        run_metrics["baseline"]["eval"] = eval_model_datasets(
            datasets=eval_datasets,
            model=model,
            device=device,
            speed_batch_size=args.speed_batch_size,
            speed_measurements_steps=args.speed_measurements_steps,
            save_features=args.save_features,
            features_output_dir=baseline_features_output_dir
        )

        baseline_metrics_dir = os.path.join(output_dir, "baseline")

        os.makedirs(baseline_metrics_dir, exist_ok=True)

        with open(os.path.join(baseline_metrics_dir, "metrics.json"), "w") as f:
            json.dump(run_metrics["baseline"]["eval"], f)

    # Load a dataset to optimize the block weights using a genetic algorithm
    ga_optimization_dataset = load_dataset(
        type_=args.ga_optimization_dataset,
        split=args.ga_optimization_dataset_split,
        cache_dir=args.cache_dir,
        transform=transform,
        max_samples=args.ga_max_eval_samples,
        keep_in_memory=args.ga_keep_in_memory
    )

    if args.use_fast_dataset:
        logging.info("precomputing the genetic algorithm optimization dataset transform")

        ga_optimization_dataset = FastDataset(ga_optimization_dataset)
        ga_optimization_dataset.precompute()

    logging.info(f"loaded the genetic algorithm optimization dataset of type {args.ga_optimization_dataset} with {len(ga_optimization_dataset)} samples")

    ga_optimization_data_loader_args = data_loader_args | { "batch_size": args.ga_batch_size }
    ga_optimization_data_loader = DataLoader(
        dataset=ga_optimization_dataset,
        **ga_optimization_data_loader_args
    )

    if args.correction_dataset is not None:
        correction_dataset = load_dataset(
            type_=args.correction_dataset,
            split=DatasetSplit.TRAIN,
            cache_dir=args.cache_dir,
            transform=transform,
            max_samples=args.correction_max_samples
        )

        correction_data_loader = DataLoader(
            dataset=correction_dataset,
            **data_loader_args | { "batch_size": args.correction_batch_size }
        )
    elif args.correction_dataset is None and args.pruning_strategy in [PrunableModelType.SPARSE_GPT]:
        raise ValueError("sparse-gpt applies weight correction, but no correction dataset was provided")
    else:
        correction_data_loader = None

    block_weights = torch.cat([
        torch.linspace(args.ga_init_start_value, args.ga_init_end_value, num_blocks),
        torch.linspace(args.ga_init_start_value, args.ga_init_end_value, num_blocks).unsqueeze(dim=1).repeat(1, num_heads).reshape(-1)
    ], dim=0).numpy()

    pruning_runtime_s = 0

    logging.info("optimizing the block weights...")

    def pruning_function(model: PrunableModel, solution: np.ndarray) -> List[PrunableModel]:
        models = []

        for mlp_ratio, head_ratio in zip(args.ga_mlp_pruning_ratios, args.ga_heads_pruning_ratios):
            prunable = deepcopy_model(model)
            prunable.prune(
                data_loader=pruning_data_loader,
                pruning_ratio=mlp_ratio,
                pruning_ratio_heads=head_ratio,
                # Take the absolute value of the solution as some entries might be negative
                block_weights=torch.tensor(solution.astype(np.float32)).abs(),
                estimate_pruning_weights=False
            )

            models.append(prunable)

        return models

    fitness_function = build_fitness_function(
        model=model,
        teacher=teacher,
        pruning_data_loader=pruning_data_loader,
        eval_data_loader=ga_optimization_data_loader,
        device=device,
        pruning_function=pruning_function,
        loss_types=args.ga_loss_types,
        num_pca_components=args.ga_num_pca_components
    )

    if args.ga_init_covariance_matrix_path is not None:
        covariance_init = np.load(args.ga_init_covariance_matrix_path)
        run_metrics["init_covariance"] = covariance_init.tolist()
    else:
        covariance_init = None
        run_metrics["init_covariance"] = None

    step_start_time = time.time()

    if args.ga_max_function_evaluations > 0:
        block_weights, block_weights_fitness, info = optimize_function_ga(
            # Generic arguments
            type_=args.ga_optimizer,
            fitness_function=fitness_function,
            ndim_problem=num_blocks + num_heads * num_blocks,
            max_function_evaluations=args.ga_max_function_evaluations,
            seed=args.seed,
            starting_point=block_weights,
            covariance_init=covariance_init,
            xnes_num_individuals=args.ga_xnes_num_individuals
        )
    else:
        block_weights, block_weights_fitness, info = block_weights, 0, {}

    model = model.to(device)

    run_metrics["fitness_scores"] = info.get("fitness_scores")
    run_metrics["final_covariance"] = info.get("covariance")

    block_weights = torch.tensor(block_weights).abs()

    if args.save_importance_scores:
        importance_scores_path = os.path.join(output_dir, "importance-scores.pt")

        logging.info(f"exporting the importance scores to {importance_scores_path}...")

        importance_scores = export_importance_scores(
            prunable_model=model,
            block_weights=block_weights,
            model_type=args.model_type,
            metadata={
                "seed": args.seed,
                "pruning_strategy": args.pruning_strategy.value,
                "pruning_dataset": args.pruning_dataset.value,
                "pruning_dataset_split": args.pruning_dataset_split.value,
                "max_samples": args.max_samples,
                "num_estimation_epochs": args.num_estimation_epochs,
                "min_hidden_dim_keep_ratio": args.min_hidden_dim_keep_ratio,
                "min_head_keep_ratio": args.min_head_keep_ratio,
                "ga_optimization_dataset": args.ga_optimization_dataset.value,
                "ga_optimization_dataset_split": args.ga_optimization_dataset_split.value,
                "ga_max_eval_samples": args.ga_max_eval_samples,
                "ga_optimizer": args.ga_optimizer.value,
                "ga_max_function_evaluations": args.ga_max_function_evaluations,
                "ga_num_pca_components": args.ga_num_pca_components,
                "ga_loss_types": [t.value for t in args.ga_loss_types],
                "ga_xnes_num_individuals": args.ga_xnes_num_individuals,
                "ga_init_start_value": args.ga_init_start_value,
                "ga_init_end_value": args.ga_init_end_value,
                "ga_mlp_pruning_ratios": args.ga_mlp_pruning_ratios,
                "ga_heads_pruning_ratios": args.ga_heads_pruning_ratios,
            }
        )

        importance_scores.save(importance_scores_path)

        logging.info(f"importance scores saved to {importance_scores_path}")

    if args.ga_save_weights:
        # Compute the base weights
        mlp_weights = model._reduce_pruning_weights()
        head_weights = model._reduce_heads_pruning_weights()

        # Rescale the weights
        scaling_block_weights = torch.tensor(block_weights).to(device)

        for i in range(num_blocks):
            mlp_weights[i] = mlp_weights[i] * scaling_block_weights[i]
            head_weights[i] = head_weights[i] * scaling_block_weights[i]

        # Save them to disk
        mlp_weights = torch.cat([
            pad_vector_to_match(
                x,
                target_length=hidden_dim,
                value=0
            ).unsqueeze(dim=0) for x in mlp_weights
        ]).cpu()

        head_weights = torch.cat([
            pad_vector_to_match(
                x,
                target_length=num_heads,
                value=0
            ).unsqueeze(dim=0) for x in head_weights
        ]).cpu()

        weights_path = os.path.join(output_dir, "ga-weights.pt")

        torch.save({
            "mlp": mlp_weights,
            "head": head_weights
        }, weights_path)

    logging.info(f"the optimized block weights (fitness {block_weights_fitness}) are {block_weights.tolist()}")

    pruning_runtime_s += time.time() - step_start_time

    logging.info(f"the total optimization runtime was {pruning_runtime_s:.2f} seconds")

    for mlp_ratio, head_ratio in zip(args.eval_mlp_pruning_ratios, args.eval_heads_pruning_ratios):
        logging.info(f"pruning the mlp with ratio {mlp_ratio} and the heads with ratio {head_ratio}")

        pruned_model_output_dir = os.path.join(output_dir, f"mlp-{mlp_ratio}-heads-{head_ratio}")
        os.makedirs(pruned_model_output_dir, exist_ok=True)
        save_masks_path = os.path.join(pruned_model_output_dir, "pruning_masks.pt")

        prunable = deepcopy_model(model)
        prunable.prune(
            data_loader=pruning_data_loader,
            pruning_ratio=mlp_ratio,
            pruning_ratio_heads=head_ratio,
            block_weights=torch.tensor(block_weights).to(device),
            estimate_pruning_weights=False,
            apply_correction=True,
            correction_data_loader=correction_data_loader,
            damping_percentage=args.correction_damping_percentage,
            damping_strategy=args.correction_damping_strategy,
	    save_masks_path=save_masks_path  # ← NOW SAFE TO PASS
        )

        # Save the model metrics
        model_metrics = {
            "num_heads": [],
            "hidden_dims": [],
            "flops": estimate_model_flops(model=prunable),
            "ratios": {
                "mlp": mlp_ratio,
                "heads": head_ratio
            }
        }

        logging.info(f"this model has {count_parameters(prunable.model)} parameters")
        logging.info(f"this model has {estimate_model_flops(model=prunable)} FLOPs")

        pruned_model_output_dir = os.path.join(output_dir, f"mlp-{mlp_ratio}-heads-{head_ratio}")
        features_output_dir = os.path.join(pruned_model_output_dir, "pruned", "features")

        logging.info(f"saving the pruned model to {pruned_model_output_dir}")

        if args.save_features:
            logging.info(f"saving the features of the pruned model to {features_output_dir}")
            os.makedirs(features_output_dir, exist_ok=True)


        if args.save_pruned_model:
          logging.info(f"saving the pruned model to {pruned_model_output_dir}")
          save_pruned_model(output_dir=pruned_model_output_dir,model=prunable,model_config=model_config)

        model_metrics["eval"] = eval_model_datasets(
            datasets=eval_datasets,
            model=prunable,
            device=device,
            speed_batch_size=args.speed_batch_size,
            speed_measurements_steps=args.speed_measurements_steps,
            save_features=args.save_features,
            features_output_dir=features_output_dir
        )

        for i in range(num_blocks):
            if hasattr(prunable.model.blocks[i].mlp, MLPLayerType.FC1.value):
                block_hidden_dim = prunable.model.blocks[i].mlp.fc1.weight.shape[0]
            elif hasattr(prunable.model.blocks[i].mlp, MLPLayerType.W1.value):
                block_hidden_dim = prunable.model.blocks[i].mlp.w1.weight.shape[0]
            else:
                raise ValueError(f"Unknown MLP architecture: {prunable.model.blocks[i].mlp}")

            block_hidden_dim_percent = round((block_hidden_dim / hidden_dim) * 100, 2)

            model_metrics["hidden_dims"].append(block_hidden_dim_percent)

            logging.info(f"the hidden dimensionality of the {i}-th block is now {block_hidden_dim}, i.e. {block_hidden_dim_percent}% of the original")

        for i in range(num_blocks):
            block_num_heads = prunable.model.blocks[i].attn.num_heads
            block_num_heads_percent = round((block_num_heads / num_heads) * 100, 2)

            model_metrics["num_heads"].append(block_num_heads_percent)

            logging.info(f"the number of heads of the {i}-th block is now {block_num_heads}, i.e. {block_num_heads_percent}% of the original")

        run_metrics["pruned"].append(model_metrics)

    # Save the metrics and the run configuration to disk
    save_run_info(
        output_dir=output_dir,
        metrics=run_metrics,
        args=args,
        model=None
    )


if __name__ == "__main__":
    parser = argparse.ArgumentParser()

    parser.add_argument("--seed", type=int, default=42, help="The seed for random number generators")
    parser.add_argument("--cache-dir", type=str, required=False, default=default_cache_dir(), help="The cache directory for datasers and models")
    parser.add_argument("--batch-size", type=int, default=16, help="The batch size")
    parser.add_argument("--num-workers", type=int, default=4, help="The number of workers for the data loader")
    parser.add_argument("--pruning-dataset", type=DatasetType, choices=list(DatasetType), default=DatasetType.IN1K, help="The dataset to use to select which weights to prune")
    parser.add_argument("--pruning-dataset-split", type=DatasetSplit, choices=list(DatasetSplit), default=DatasetSplit.TEST, help="The split of the dataset to use for pruning")
    parser.add_argument("--eval-datasets", type=DatasetType, nargs='+', choices=list(DatasetType), default=[DatasetType.IN1K], help="The datasets to use for evaluation, can specify multiple datasets")
    parser.add_argument("--max-samples", type=int, default=None, help="The maximum number of samples to use in estimating the prunable weights. If larger than the dataset size, all samples are used.")
    parser.add_argument("--model-type", type=ModelType, choices=list(ModelType), default=ModelType.AUGREG_VIT_S_16_IN21K_FT_IN1K, help="The model to be pruned")
    parser.add_argument("--use-fast-dataset", action=argparse.BooleanOptionalAction, default=False, help="Whether to use the fast dataset")
    parser.add_argument("--pruning-strategy", type=PrunableModelType, choices=list(PrunableModelType), default=PrunableModelType.CROSS_ENTROPY, help="The type of pruning strategy to use")
    parser.add_argument("--num-estimation-epochs", type=int, default=1, help="The number of epochs to use for estimating the gradients")
    parser.add_argument("--min-hidden-dim-keep-ratio", type=float, default=0.2, help="The minimum ratio of hidden neurons to keep in a block's mlp")
    parser.add_argument("--min-head-keep-ratio", type=float, default=0.2, help="The minimum ratio of heads to keep in a block's attention layer")
    parser.add_argument("--speed-measurements-steps", type=int, default=500, help="The number of steps to run for measuring the model speed")
    parser.add_argument("--speed-batch-size", type=int, default=16, help="The batch size to use for measuring the model speed")
    parser.add_argument("--max-knn-train-samples", type=int, default=None, help="The maximum number of samples to use for the knn evaluation. If larger than the dataset size or None, all samples are used.")
    parser.add_argument("--eval-baseline", action=argparse.BooleanOptionalAction, default=False, help="Whether to evaluate the baseline model before pruning")
    parser.add_argument("--save-pruned-model", action=argparse.BooleanOptionalAction, default=False, help="Whether to save the pruned model")
    parser.add_argument("--output-dir", type=str, required=True, help="Where to save checkpoints, metrics and logs")
    parser.add_argument("--save-features", action=argparse.BooleanOptionalAction, default=False, help="Whether to save the features of the pruned model")
    parser.add_argument("--save-importance-scores", action=argparse.BooleanOptionalAction, default=False, help="Whether to save importance scores for elastic inference")
    parser.add_argument("--ga-max-eval-samples", type=int, default=512, help="The maximum number of samples for evaluating the fitness of the genetic algorithm solutions")
    parser.add_argument("--ga-optimization-dataset", type=DatasetType, choices=list(DatasetType), default=DatasetType.IN1K, help="The dataset to use for the genetic algorithm optimization")
    parser.add_argument("--ga-optimization-dataset-split", type=DatasetSplit, choices=list(DatasetSplit), default=DatasetSplit.TEST, help="The split of the dataset to use for the genetic algorithm optimization")
    parser.add_argument("--ga-keep-in-memory", action=argparse.BooleanOptionalAction, default=False, help="Whether to load the genetic algorithm optimization dataset in memory")
    parser.add_argument("--ga-batch-size", type=int, default=16, help="The batch size to use for the genetic algorithm optimization")
    parser.add_argument("--ga-optimizer", type=GAOptimizerType, choices=list(GAOptimizerType), default=GAOptimizerType.XNES, help="The optimizer to use for the genetic algorithm")
    parser.add_argument("--ga-max-function-evaluations", type=int, default=250, help="The maximum number of function evaluations for the genetic algorithm")
    parser.add_argument("--ga-num-pca-components", type=int, default=64, help="The number of PCA components to use for the genetic algorithm")
    parser.add_argument("--ga-xnes-num-individuals", type=int, default=None, help="The number of individuals to use for the xnes genetic algorithm")
    parser.add_argument("--ga-loss-types", type=GALossType, nargs='+', choices=list(GALossType), default=[GALossType.COSINE_SIMILARITY], help="The loss types to use for the genetic algorithm")
    parser.add_argument("--ga-save-weights", action=argparse.BooleanOptionalAction, default=False, help="Whether to save the weights of the genetic algorithm")
    parser.add_argument("--ga-init-start-value", type=float, default=1.2, help="The initial genetic algorithm coefficient for the first model block")
    parser.add_argument("--ga-init-end-value", type=float, default=0.8, help="The initial genetic algorithm coefficient for the last model block")
    parser.add_argument("--ga-init-covariance-matrix-path", type=str, default=None, help="The covariance matrix to use for the genetic algorithm")
    parser.add_argument("--ga-mlp-pruning-ratios", type=float, nargs="+", default=[0.25, 0.45, 0.65], help="The ratios to use for the mlp pruning in the genetic algorithm")
    parser.add_argument("--ga-heads-pruning-ratios", type=float, nargs="+", default=[0.1, 0.3, 0.5], help="The ratios to use for the heads pruning in the genetic algorithm")
    parser.add_argument("--eval-mlp-pruning-ratios", type=float, nargs="+", default=[], help="The ratios to use for the mlp pruning in the genetic algorithm")
    parser.add_argument("--eval-heads-pruning-ratios", type=float, nargs="+", default=[], help="The ratios to use for the heads pruning in the genetic algorithm")
    parser.add_argument("--correction-dataset", type=DatasetType, choices=list(DatasetType), default=None, help="The dataset to use for weight correction")
    parser.add_argument("--correction-max-samples", type=int, default=None, help="The maximum number of samples to use for weight correction. If larger than the dataset size, all samples are used.")
    parser.add_argument("--correction-batch-size", type=int, default=16, help="The batch size to use for weight correction")
    parser.add_argument("--correction-damping-percentage", type=float, default=0.01, help="Damping percentage for SparseGPT weight correction")
    parser.add_argument("--correction-damping-strategy", type=SparseGPTDampingStrategy, choices=list(SparseGPTDampingStrategy), default=SparseGPTDampingStrategy.MEAN, help="Damping strategy for SparseGPT weight correction")
    configure_logging()
    main(parser.parse_args())

## logging.py : Utility functions for logging run information and saving models.

Changes:
- Updated save_pruned_model() to save both state_dict and the full pruned model object.
- This preserves reduced dimensions (num_heads, hidden_dim) after structured pruning.

In [ ]:
# @title
%%writefile SnapViT/src/utils/logging.py
import os
import sys
import time
import json
import torch
import logging
import argparse
import torch.nn as nn

from typing import Optional

from src.utils.misc import serialize_args
from src.models.prunable import PrunableModel


def configure_logging():
    logging.basicConfig(
        format="[%(asctime)s:%(levelname)s]: %(message)s",
        level=logging.INFO,
        handlers=[
            logging.FileHandler(f"{int(time.time())}.out"),
            logging.StreamHandler(sys.stdout)
        ]
    )


def save_run_info(
    output_dir: str,
    metrics: dict,
    args: argparse.Namespace,
    model: Optional[nn.Module | PrunableModel] = None
):
    try:
        with open(os.path.join(output_dir, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=4)
    except Exception as e:
        logging.error(f"error saving the metrics to disk: {e}")
        logging.error(f"the metrics were: {metrics}")

    try:
        with open(os.path.join(output_dir, "args.json"), "w") as f:
            json.dump(serialize_args(args=args), f, indent=4)
    except Exception as e:
        logging.error(f"error saving the args to disk: {e}")
        logging.error(f"the args were: {args}")

    # Save the model to disk if it's not None
    if model is not None:
        save_pruned_model(output_dir=output_dir, model=model)


def save_pruned_model(
    output_dir: str,
    model: nn.Module | PrunableModel,
    model_config: Optional[dict] = None
):
    pruned_vit = model.model if isinstance(model, PrunableModel) else model

    torch.save(pruned_vit.state_dict(), os.path.join(output_dir, "pruned-model.pt"))
    full_path = os.path.join(output_dir, "full_pruned_model.pt")
    torch.save(pruned_vit, full_path)

    total_params = sum(p.numel() for p in pruned_vit.parameters())
    heads_block0 = pruned_vit.blocks[0].attn.num_heads
    mlp_dim_block0 = pruned_vit.blocks[0].mlp.fc1.out_features

    logging.info(f"PRUNING SUCCESSFUL → Saved truly pruned model")
    logging.info(f"Parameters: {total_params:,} (should be ~34–38M, not 86M)")
    logging.info(f"Block-0 heads: {heads_block0}, MLP hidden dim: {mlp_dim_block0}")

## base.py : Base class for prunable models.

Changes vs original SnapViT:
   1. MLP pruning granularity changed from per-neuron to group-of-8:
      neurons are ranked and pruned in aligned blocks of 8, matching the
      8-wide inner-loop granularity of TVM-generated dense kernels.
   2. Group score aggregation uses mean (not min) importance across the
      8 neurons in each group, so one weak neuron does not drag down an
      otherwise important group.
   3. Min-keep protection applied at group level — the minimum fraction
      of neurons to preserve per block is enforced in whole groups.
   4. Added save_masks_path parameter to prune(): when provided, binary
      MLP and head masks are saved as a .pt file after each pruning call,
      giving the group-aligned masks needed by the RISC-V runtime.

In [ ]:
# @title
%%writefile SnapViT/src/models/prunable/base.py
import torch
import logging
import torch.nn as nn
import os

from torch.utils.data import DataLoader
from typing import Optional, List, Tuple

from src.models.enums import MLPLayerType
from src.utils.models import predict, block_num_heads

MLP_GROUP_SIZE = 8  # ← SIMD group width;


class PrunableModel(nn.Module):
    def __init__(
        self,
        model: nn.Module,
        device: torch.device,
        min_hidden_dim_keep_ratio: float = 0.2,
        min_head_keep_ratio: float = 0.2,
        **kwargs
    ):
        super().__init__()

        self.model = model
        self.device = device
        self.head_dim = model.embed_dim // model.blocks[0].attn.num_heads
        self.default_num_heads = model.blocks[0].attn.num_heads
        self.min_head_keep_count = int(self.default_num_heads * min_head_keep_ratio)
        self.min_head_keep_ratio = min_head_keep_ratio
        self.num_blocks = len(self.model.blocks)
        self.min_hidden_dim_keep_ratio = min_hidden_dim_keep_ratio

        if hasattr(self.model.blocks[0].mlp, MLPLayerType.FC1.value):
            self.target_input_mlp_layer = MLPLayerType.FC1
            self.target_output_mlp_layer = MLPLayerType.FC2
        elif hasattr(self.model.blocks[0].mlp, MLPLayerType.W1.value):
            self.target_input_mlp_layer = MLPLayerType.W1
            self.target_output_mlp_layer = MLPLayerType.W3
        else:
            raise ValueError(f"Unknown MLP architecture: {self.model.blocks[0].mlp}")

        self.embeddings_dim = getattr(self.model.blocks[0].mlp, self.target_output_mlp_layer.value).out_features
        self.default_mlp_hidden_dim = getattr(self.model.blocks[0].mlp, self.target_input_mlp_layer.value).out_features

        # GROUP PRUNING
        # hidden_dim must be divisible by MLP_GROUP_SIZE, If not , fall back to G=1.
        if self.default_mlp_hidden_dim % MLP_GROUP_SIZE == 0:
            self.mlp_group_size = MLP_GROUP_SIZE
        else:
            logging.warning(
                f"hidden_dim {self.default_mlp_hidden_dim} is not divisible by "
                f"MLP_GROUP_SIZE={MLP_GROUP_SIZE}. Falling back to group_size=1 "
                f"(original per-neuron behaviour)."
            )
            self.mlp_group_size = 1

        self.num_mlp_groups = self.default_mlp_hidden_dim // self.mlp_group_size

        self.min_hidden_dim_keep_count = max(
            1,
            int(self.default_mlp_hidden_dim * min_hidden_dim_keep_ratio) // self.mlp_group_size
        )
        # ──────────────────────────────────────────────────────────────────────

    @property
    def blocks(self) -> List[nn.Module]:
        return self.model.blocks

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        return self.model.forward_features(x)

    def estimate_pruning_weights(self, data_loader: DataLoader, backward: bool = True):
        """Estimate the model pruning weights using a cross-entropy loss."""
        predict(model=self.model, data_loader=data_loader, device=self.device, backward=backward)

    def prune(
        self,
        data_loader: DataLoader,
        pruning_ratio: float,
        pruning_ratio_heads: float,
        block_weights: Optional[torch.Tensor] = None,
        estimate_pruning_weights: bool = True,
        save_masks_path: Optional[str] = None,
        **kwargs
    ):
        if estimate_pruning_weights:
            self.estimate_pruning_weights(data_loader=data_loader)

        if save_masks_path is not None:
            self.all_mlp_masks = []
            self.all_head_masks = []

        hidden_dims = [
            getattr(block.mlp, self.target_input_mlp_layer.value).bias.shape[0]
            for block in self.model.blocks
        ]
        head_dims = [block_num_heads(block=block) for block in self.model.blocks]

        pruning_weights, head_pruning_weights = self.compute_pruning_weights(
            pruning_ratio=pruning_ratio,
            pruning_ratio_heads=pruning_ratio_heads,
            block_weights=block_weights
        )

        block_offset, head_offset = 0, 0
        for i, (hidden_dim, num_heads) in enumerate(zip(hidden_dims, head_dims)):
            self.prune_block_mlp(
                block_index=i,
                pruning_weights=pruning_weights[block_offset:block_offset + hidden_dim],
                hidden_dim=hidden_dim
            )
            self.prune_block_attn(
                block_index=i,
                pruning_weights=head_pruning_weights[head_offset:head_offset + num_heads],
                num_heads=num_heads
            )
            block_offset += hidden_dim
            head_offset += num_heads

        if save_masks_path is not None:
            torch.save({
                'mlp_masks': torch.stack(self.all_mlp_masks),    # [num_blocks, hidden_dim]
                'head_masks': torch.stack(self.all_head_masks),  # [num_blocks, num_heads]
                'pruning_ratio_mlp': pruning_ratio,
                'pruning_ratio_heads': pruning_ratio_heads,
                'original_hidden_dim': self.default_mlp_hidden_dim,
                'original_num_heads': self.default_num_heads,
                'mlp_group_size': self.mlp_group_size           # record G for downstream use
            }, save_masks_path)
            logging.info(f"Pruning masks saved to: {save_masks_path}")

    def compute_pruning_weights(
        self,
        pruning_ratio: float,
        pruning_ratio_heads: float,
        block_weights: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:

        G = self.mlp_group_size

        # ── 1. Collect raw per-neuron and per-head scores ──────────────────────
        pruning_weights = self._reduce_pruning_weights()    # list of [hidden_dim] per block
        head_pruning_weights = self._reduce_heads_pruning_weights()  # list of [num_heads] per block

        # ── 2. Apply GA block weights and protect min-keep neurons ─────────────
        if block_weights is not None:
            block_weights = block_weights.to(self.device)
            mlp_weights = block_weights[:self.num_blocks]
            head_weights = block_weights[self.num_blocks:].reshape(self.num_blocks, -1)

            for i in range(self.num_blocks):
                pruning_weights[i] = pruning_weights[i] * mlp_weights[i]
                head_pruning_weights[i] = head_pruning_weights[i] * head_weights[i]

                # ── GROUP-LEVEL min-keep protection ────────────────────────────
                # Aggregate scores to group level, find the top-k groups to protect,
                # then set ALL neurons in those groups to max so they are never pruned.
                if G > 1:

                    group_scores_i = pruning_weights[i].view(-1, G).mean(dim=1)
                    top_group_indices = group_scores_i.argsort(descending=True)[:self.min_hidden_dim_keep_count]

                    neuron_indices_to_protect = (
                        top_group_indices.unsqueeze(1) * G + torch.arange(G, device=self.device)
                    ).flatten()
                    pruning_weights[i][neuron_indices_to_protect] = torch.finfo(pruning_weights[i].dtype).max
                else:

                    sorted_indices = pruning_weights[i].argsort(descending=True)
                    pruning_weights[i][sorted_indices[:self.min_hidden_dim_keep_count]] = (
                        torch.finfo(pruning_weights[i].dtype).max
                    )

                sorted_head_indices = head_pruning_weights[i].argsort(descending=True)
                head_pruning_weights[i][sorted_head_indices[:self.min_head_keep_count]] = (
                    torch.finfo(head_pruning_weights[i].dtype).max
                )

        #  3. Concatenate across blocks
        pruning_weights = torch.cat(pruning_weights, dim=0)
        head_pruning_weights = torch.cat(head_pruning_weights, dim=0)

        # 4. GROUP-LEVEL global ranking for MLP

        total_neurons = pruning_weights.shape[0]
        total_groups  = total_neurons // G

        group_scores = pruning_weights.view(total_groups, G).mean(dim=1)


        group_scores_indices = group_scores.argsort(descending=False)

        num_total_pruned_neurons = int(self.default_mlp_hidden_dim * pruning_ratio * self.num_blocks)

        num_total_pruned_groups = num_total_pruned_neurons // G

        logging.info(
            f"pruning {num_total_pruned_groups} groups "
            f"({num_total_pruned_groups * G} neurons, G={G})..."
        )


        groups_to_prune = group_scores_indices[:num_total_pruned_groups]
        neurons_to_prune = (
            groups_to_prune.unsqueeze(1) * G + torch.arange(G, device=pruning_weights.device)
        ).flatten()
        pruning_weights[neurons_to_prune] = 0

        num_total_pruned_heads = int(self.default_num_heads * pruning_ratio_heads * self.num_blocks)
        head_weights_indices = head_pruning_weights.argsort(descending=False)
        heads_to_prune = head_weights_indices[:num_total_pruned_heads]

        logging.info(f"pruning {num_total_pruned_heads} attention heads...")

        head_pruning_weights[heads_to_prune] = 0

        return pruning_weights, head_pruning_weights

    def prune_block_mlp(self, block_index: int, pruning_weights: torch.tensor, hidden_dim: int):
        if self.target_input_mlp_layer == MLPLayerType.FC1:
            self.prune_block_standard_mlp(
                block_index=block_index,
                pruning_weights=pruning_weights,
                hidden_dim=hidden_dim
            )
        elif self.target_input_mlp_layer == MLPLayerType.W1:
            self.prune_block_swiglu_mlp(
                block_index=block_index,
                pruning_weights=pruning_weights,
                hidden_dim=hidden_dim
            )
        else:
            raise ValueError(f"Unknown MLP architecture: {self.model.blocks[block_index].mlp}")

    def prune_block_standard_mlp(self, block_index: int, pruning_weights: torch.tensor, hidden_dim: int):
        # Sort pruning weights so zeroed (pruned) neurons are at the end
        sorting_indices = pruning_weights.argsort(dim=-1, descending=True)

        num_zeros = len(torch.where(pruning_weights == 0)[0])
        num_neurons = hidden_dim - num_zeros


        if hasattr(self, 'all_mlp_masks'):
            mask = torch.ones(hidden_dim)
            mask[sorting_indices[num_neurons:]] = 0
            self.all_mlp_masks.append(mask.cpu())


        self.model.blocks[block_index].mlp.fc1.weight.data = (
            self.model.blocks[block_index].mlp.fc1.weight.data[sorting_indices, :]
        )
        self.model.blocks[block_index].mlp.fc2.weight.data = (
            self.model.blocks[block_index].mlp.fc2.weight.data[:, sorting_indices]
        )
        self.model.blocks[block_index].mlp.fc1.bias.data = (
            self.model.blocks[block_index].mlp.fc1.bias[sorting_indices]
        )

        self.model.blocks[block_index].mlp.fc1.weight.data = (
            self.model.blocks[block_index].mlp.fc1.weight.data[:num_neurons, :]
        )
        self.model.blocks[block_index].mlp.fc2.weight.data = (
            self.model.blocks[block_index].mlp.fc2.weight.data[:, :num_neurons]
        )
        self.model.blocks[block_index].mlp.fc1.bias.data = (
            self.model.blocks[block_index].mlp.fc1.bias[:num_neurons]
        )

    def prune_block_swiglu_mlp(self, block_index: int, pruning_weights: torch.tensor, hidden_dim: int):
        sorting_indices = pruning_weights.argsort(dim=-1, descending=True)

        num_zeros = len(torch.where(pruning_weights == 0)[0])
        num_neurons = hidden_dim - num_zeros

        # ── SAVE MLP MASK ──────────────────────────────────────────────────────
        if hasattr(self, 'all_mlp_masks'):
            mask = torch.ones(hidden_dim)
            mask[sorting_indices[num_neurons:]] = 0
            self.all_mlp_masks.append(mask.cpu())
        # ──────────────────────────────────────────────────────────────────────

        self.model.blocks[block_index].mlp.w1.weight.data = (
            self.model.blocks[block_index].mlp.w1.weight.data[sorting_indices, :]
        )
        self.model.blocks[block_index].mlp.w2.weight.data = (
            self.model.blocks[block_index].mlp.w2.weight.data[sorting_indices, :]
        )
        self.model.blocks[block_index].mlp.w3.weight.data = (
            self.model.blocks[block_index].mlp.w3.weight.data[:, sorting_indices]
        )
        self.model.blocks[block_index].mlp.w1.bias.data = (
            self.model.blocks[block_index].mlp.w1.bias[sorting_indices]
        )
        self.model.blocks[block_index].mlp.w2.bias.data = (
            self.model.blocks[block_index].mlp.w2.bias[sorting_indices]
        )

        self.model.blocks[block_index].mlp.w1.weight.data = (
            self.model.blocks[block_index].mlp.w1.weight.data[:num_neurons, :]
        )
        self.model.blocks[block_index].mlp.w2.weight.data = (
            self.model.blocks[block_index].mlp.w2.weight.data[:num_neurons, :]
        )
        self.model.blocks[block_index].mlp.w3.weight.data = (
            self.model.blocks[block_index].mlp.w3.weight.data[:, :num_neurons]
        )
        self.model.blocks[block_index].mlp.w1.bias.data = (
            self.model.blocks[block_index].mlp.w1.bias[:num_neurons]
        )
        self.model.blocks[block_index].mlp.w2.bias.data = (
            self.model.blocks[block_index].mlp.w2.bias[:num_neurons]
        )

    def prune_block_attn(self, block_index: int, pruning_weights: torch.tensor, num_heads: int):

        sorting_indices = pruning_weights.argsort(dim=-1, descending=True)

        num_zeros = len(torch.where(pruning_weights == 0)[0])
        num_remaining_heads = max(1, num_heads - num_zeros)

        # === SAVE HEAD MASK ===
        if hasattr(self, 'all_head_masks'):
            mask = torch.ones(num_heads)
            mask[sorting_indices[num_remaining_heads:]] = 0
            self.all_head_masks.append(mask.cpu())

        block = self.model.blocks[block_index]

        out_weights = block.attn.proj.weight.reshape(
            self.embeddings_dim,
            num_heads,
            self.head_dim
        )[:, sorting_indices, :]

        block.attn.proj.weight.data = out_weights[:, :num_remaining_heads, :].reshape(
            self.embeddings_dim,
            -1
        )

        qkv_weights = block.attn.qkv.weight.view(3, num_heads, self.head_dim, self.embeddings_dim)
        qkv_weights = qkv_weights[:, sorting_indices, :, :]
        qkv_weights = qkv_weights[:, :num_remaining_heads, :, :]
        qkv_weights = qkv_weights.reshape(-1, self.embeddings_dim)

        block.attn.qkv.weight.data = qkv_weights

        self.prune_qkv_bias(
            block=block,
            sorting_indices=sorting_indices,
            num_heads=num_heads,
            num_remaining_heads=num_remaining_heads,
            bias_key="bias"
        )
        self.prune_qkv_bias(
            block=block,
            sorting_indices=sorting_indices,
            num_heads=num_heads,
            num_remaining_heads=num_remaining_heads,
            bias_key="bias_mask"
        )

        for key in ["q_bias", "k_bias", "v_bias"]:
            if not hasattr(block.attn, key):
                continue
            if getattr(block.attn, key) is None:
                continue
            qkv_bias = getattr(block.attn, key).view(num_heads, self.head_dim)
            qkv_bias = qkv_bias[sorting_indices, :]
            qkv_bias = qkv_bias[:num_remaining_heads, :]
            qkv_bias = qkv_bias.reshape(-1)
            getattr(block.attn, key).data = qkv_bias

        block.attn.num_heads = num_remaining_heads

    def prune_qkv_bias(self, block, sorting_indices, num_heads, num_remaining_heads, bias_key):
        if hasattr(block.attn.qkv, bias_key) and getattr(block.attn.qkv, bias_key) is not None:
            bias = getattr(block.attn.qkv, bias_key).view(3, num_heads, self.head_dim)
            bias = bias[:, sorting_indices, :]
            bias = bias[:, :num_remaining_heads]
            bias = bias.reshape(-1)
            getattr(block.attn.qkv, bias_key).data = bias

    def _reduce_pruning_weights(self) -> List[torch.Tensor]:
        return [
            (getattr(block.mlp, self.target_input_mlp_layer.value).weight.grad ** 2).mean(dim=1)
            for block in self.model.blocks
        ]

    def _reduce_heads_pruning_weights(self) -> List[torch.Tensor]:
        reduced = []
        for block in self.model.blocks:
            num_heads = block_num_heads(block=block)
            head_grads = block.attn.qkv.weight.grad.reshape(
                3,
                num_heads,
                self.head_dim,
                self.embeddings_dim
            )[2, :, :, :] ** 2
            head_grads = head_grads.mean(dim=(1, 2))
            reduced.append(head_grads)
        return reduced

## dino.sh : configure and run multi-model GA pruning

Changes:
- eval_dataset changed to 'pets'
- changed 'cache_dir'
- added multiple models to 'MODEL_VALUES'

In [ ]:
# @title
%%writefile SnapViT/jobs/ga/dino.sh
MIN_HEAD_RATIO=0.2
MIN_HIDDEN_DIM_RATIO=0.05

# Prunability score estimation parameters
PRUNING_STRATEGY=dino
NUM_SAMPLES=1000
ESTIMATION_EPOCHS=1
BATCH_SIZE=16
NUM_WORKERS=8

# Pruning and evaluation datasets
PRUNING_DATASET_SPLIT=train
PRUNING_DATASET_NAME=pets

EVAL_DATASET_NAMES="pets"

# Genetic algorithm parameters
GA_OPTIMIZER=xnes
GA_BATCH_SIZE=64
GA_NUM_PCA_COMPONENTS=192
GA_MAX_EVAL_SAMPLES=1000
GA_MAX_FUNCTION_EVALUATIONS=50
GA_OPTIMIZATION_DATASET=pets
GA_OPTIMIZATION_DATASET_SPLIT=train
GA_LOSS_TYPES=cosine-similarity

CACHE_DIR=/content/cache
OUTPUT_DIR=/content/pruning/artifacts/ga

mkdir -p $OUTPUT_DIR

#Names of the model families to prune
MODEL_VALUES=(
     "augreg-vit-s-16-in21k-ft-in1k"
    "dino_vits16"
    "augreg-vit-b-16-in21k-ft-in1k"
    "dino_vitb16"
)

MLP_PRUNING_RATIOS="0.15 0.35 0.55 0.65"
HEAD_PRUNING_RATIOS="0 0.2 0.4 0.5"

EVAL_MLP_PRUNING_RATIOS="0.15 0.25 0.35 0.45 0.55 0.65"
EVAL_HEAD_PRUNING_RATIOS="0 0.1 0.2 0.3 0.4 0.5"

# This loop will run model x ratios iterations
for MODEL in ${MODEL_VALUES[@]}; do
    python prune.py \
        --pruning-dataset $PRUNING_DATASET_NAME \
        --pruning-dataset-split $PRUNING_DATASET_SPLIT \
        --eval-datasets $EVAL_DATASET_NAMES \
        --cache-dir $CACHE_DIR \
        --seed 42 \
        --batch-size $BATCH_SIZE \
        --num-workers $NUM_WORKERS \
        --max-samples $NUM_SAMPLES \
        --model-type $MODEL \
        --num-estimation-epochs $ESTIMATION_EPOCHS \
        --pruning-strategy $PRUNING_STRATEGY \
        --min-hidden-dim-keep-ratio $MIN_HIDDEN_DIM_RATIO \
        --min-head-keep-ratio $MIN_HEAD_RATIO \
        --ga-max-eval-samples $GA_MAX_EVAL_SAMPLES \
        --ga-optimization-dataset $GA_OPTIMIZATION_DATASET \
        --ga-optimization-dataset-split $GA_OPTIMIZATION_DATASET_SPLIT \
        --ga-keep-in-memory \
        --ga-batch-size $GA_BATCH_SIZE \
        --ga-optimizer $GA_OPTIMIZER \
        --ga-max-function-evaluations $GA_MAX_FUNCTION_EVALUATIONS \
        --ga-num-pca-components $GA_NUM_PCA_COMPONENTS \
        --ga-loss-types $GA_LOSS_TYPES \
        --ga-mlp-pruning-ratios $MLP_PRUNING_RATIOS \
        --ga-heads-pruning-ratios $HEAD_PRUNING_RATIOS \
        --eval-mlp-pruning-ratios $EVAL_MLP_PRUNING_RATIOS \
        --eval-heads-pruning-ratios $EVAL_HEAD_PRUNING_RATIOS \
        --output-dir $OUTPUT_DIR \
        --save-pruned-model \
        --ga-save-weights \
        --save-features \
        --save-pruned-model
done

## Run pruning

In [ ]:
!cd SnapViT && ./jobs/ga/dino.sh

## Download pruned models and masks

In [ ]:
import os
import shutil

for root, dirs, files_in_dir in os.walk(base_dir):

    # Skip any directory named 'pruned'
    dirs[:] = [d for d in dirs if d != "pruned"]

    for file in files_in_dir:
        # Exclude unwanted files
        if (
            file.endswith(".json") or
            file == "ga-weights.pt" or
            file == "pruned-model.pt"
        ):
            continue

        full_path = os.path.join(root, file)
        rel_path = os.path.relpath(full_path, base_dir)
        dest_path = os.path.join(BASE_PATH, rel_path)

        # Create destination directory if needed
        os.makedirs(os.path.dirname(dest_path), exist_ok=True)

        # Copy file
        shutil.copy2(full_path, dest_path)

print(f"Files copied to: {BASE_PATH}")